# Context rot — live-model variant (#349)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgenio/contextweaver/blob/main/notebooks/context_rot_live.ipynb)

This is the **optional, real-model** companion to the deterministic demo in
[`docs/context_rot.md`](https://github.com/dgenio/contextweaver/blob/main/docs/context_rot.md)
(`scripts/context_rot_demo.py`).

The deterministic script measures a *routing-visibility* proxy (does the right
tool survive into the bounded shortlist) and runs in CI with no API key. This
notebook measures the thing the script deliberately cannot: **end-task answer
accuracy with a real model**, under growing distractor context, with and
without contextweaver.

It is **not** run in CI — it needs an OpenAI-compatible endpoint and an API
key, and its numbers are not reproducible across models/dates. Pin the model
and date when you publish a result. Tracked toward the public real-model
quality + cost benchmark in [issue #345](https://github.com/dgenio/contextweaver/issues/345).

In [ ]:
# Setup. In Colab, uncomment:
# !pip install -q contextweaver matplotlib
import os

# OpenAI-compatible endpoint (matches the repo's [e2e-eval] urllib convention
# — no vendor SDK required). Set these before running.
API_BASE = os.environ.get("CW_LLM_BASE", "https://api.openai.com/v1")
API_KEY = os.environ.get("CW_LLM_KEY", "")
MODEL = os.environ.get("CW_LLM_MODEL", "gpt-4o-mini")
DATE_PIN = "set-me-when-publishing"  # e.g. 2026-05-31
assert API_KEY, "Set CW_LLM_KEY (and optionally CW_LLM_BASE / CW_LLM_MODEL)."

In [ ]:
import json
import urllib.request

from contextweaver.eval import EvalDataset
from contextweaver.routing.catalog import generate_sample_catalog, load_catalog_dicts
from contextweaver.routing.router import Router
from contextweaver.routing.tree import TreeBuilder
from contextweaver.types import SelectableItem


def ask_model(prompt: str) -> str:
    """Single-turn completion against an OpenAI-compatible /chat/completions."""
    body = json.dumps(
        {"model": MODEL, "temperature": 0, "messages": [{"role": "user", "content": prompt}]}
    ).encode()
    req = urllib.request.Request(
        f"{API_BASE}/chat/completions",
        data=body,
        headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req) as resp:  # noqa: S310 (trusted, user-set endpoint)
        payload = json.loads(resp.read())
    return payload["choices"][0]["message"]["content"].strip()

In [ ]:
# Build a catalog of size n: the full natural pool plus seeded distractor
# variants (mirrors scripts/context_rot_demo.py:_grow_catalog), so the gold
# query set stays reachable while distractor context grows.
dataset = EvalDataset.load("benchmarks/routing_gold.json")
natural = sorted(load_catalog_dicts(generate_sample_catalog(n=10_000)), key=lambda i: i.id)
natural_ids = {i.id for i in natural}
cases = [c for c in dataset.cases if set(c.expected) & natural_ids][:40]  # keep cost bounded


def grow_catalog(n: int) -> list[SelectableItem]:
    if n <= len(natural):
        return sorted(natural, key=lambda i: i.id)[:n]
    items = list(natural)
    version = 2
    while len(items) < n:
        for orig in list(natural):
            items.append(
                SelectableItem(
                    f"{orig.id}.v{version}",
                    orig.kind,
                    f"{orig.name}_v{version}",
                    f"{orig.description} (variant {version})",
                    tags=orig.tags,
                    namespace=orig.namespace,
                )
            )
            if len(items) >= n:
                break
        version += 1
    return sorted(items, key=lambda i: i.id)[:n]


def accuracy(visible_ids: list[str], query: str, expected: set[str]) -> int:
    prompt = (
        "Pick the single best tool id for the request. Reply with the id only.\n"
        f"Tools: {', '.join(visible_ids)}\nRequest: {query}"
    )
    return int(ask_model(prompt) in expected)

In [ ]:
# For each catalog size, ask the model to pick the single best tool id for each
# gold query, two ways: (A) naive — every tool id in the grown catalog in the
# prompt; (B) contextweaver — only the routed ChoiceCard ids. Score exact-match
# tool-selection accuracy. As n grows, naive carries more distractors while
# contextweaver stays bounded.
results = []
for n in (83, 332, 1328):
    items = grow_catalog(n)
    router = Router(TreeBuilder().build(items), items=items, top_k=5)
    naive_ids = [i.id for i in items]
    naive_hits = cw_hits = 0
    for c in cases:
        expected = set(c.expected)
        naive_hits += accuracy(naive_ids, c.query, expected)
        cw_hits += accuracy(router.route(c.query).candidate_ids, c.query, expected)
    results.append((n, naive_hits / len(cases), cw_hits / len(cases)))
    print(n, results[-1])

In [ ]:
import matplotlib.pyplot as plt

sizes = [r[0] for r in results]
plt.plot(sizes, [r[1] for r in results], "o-", label="naive (all tools)")
plt.plot(sizes, [r[2] for r in results], "o-", label="contextweaver (ChoiceCards)")
plt.xscale("log")
plt.xlabel("catalog size (tools)")
plt.ylabel("tool-selection accuracy")
plt.title(f"Context rot — live model ({MODEL}, {DATE_PIN})")
plt.legend()
plt.ylim(0, 1)
plt.savefig("context_rot_live.png", dpi=120, bbox_inches="tight")
plt.show()